In [2]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET = ROOT / "dataset"
TRAIN = DATASET / "train"
TEST = DATASET / "test"

FILES = {
    "train_source1": TRAIN / "train_source1.tsv",
    "train_source2": TRAIN / "train_source2.tsv",
    "train_source3": TRAIN / "train_source3.tsv",
    "train_ground_truth": TRAIN / "train_ground_truth.tsv",
    "test_source1": TEST / "test_source1.tsv",
    "test_source2": TEST / "test_source2.tsv",
    "test_source3": TEST / "test_source3.tsv",
}

con = duckdb.connect()

print("Project root:", ROOT)
print("DuckDB:", duckdb.__version__)

Project root: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource
DuckDB: 1.5.5


In [3]:
from src.normalize import (
    normalize_name,
    normalize_address,
    normalize_country,
)


# Phase 3 — Conservative Normalization

Goal: create additional normalized representations without modifying raw data,
then measure whether normalization improves true-pair similarity.

In [4]:
examples = [
    "ABC Pvt. Ltd.",
    "  ABC   Pvt. Ltd.  ",
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "LLC Moncada Léarning Center",
    None,
]

for x in examples:
    print("RAW :", repr(x))
    print("NORM:", repr(normalize_name(x)))
    print()

RAW : 'ABC Pvt. Ltd.'
NORM: 'abc pvt. ltd.'

RAW : '  ABC   Pvt. Ltd.  '
NORM: 'abc pvt. ltd.'

RAW : 'राम मार्केटिंग प्राइवेट लिमिटेड'
NORM: 'राम मार्केटिंग प्राइवेट लिमिटेड'

RAW : 'LLC Moncada Léarning Center'
NORM: 'llc moncada léarning center'

RAW : None
NORM: None



In [5]:
sample_query = f"""
WITH sampled_gt AS (
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM read_csv_auto(
        '{FILES["train_ground_truth"].as_posix()}',
        delim='\t'
    )
    WHERE matched_entity_ids IS NOT NULL
      AND TRIM(matched_entity_ids) <> ''
    ORDER BY random()
    LIMIT 20000
),

pairs AS (
    SELECT
        source1_entity_id,
        TRIM(t.matched_entity_id) AS matched_entity_id
    FROM sampled_gt
    CROSS JOIN UNNEST(
        string_split(matched_entity_ids, ',')
    ) AS t(matched_entity_id)
),

targets AS (
    SELECT
        entity_id,
        business_name,
        business_address
    FROM read_csv_auto(
        '{FILES["train_source2"].as_posix()}',
        delim='\t'
    )

    UNION ALL

    SELECT
        entity_id,
        business_name,
        business_address
    FROM read_csv_auto(
        '{FILES["train_source3"].as_posix()}',
        delim='\t'
    )
)

SELECT
    p.source1_entity_id,
    p.matched_entity_id,
    s1.business_name AS s1_name,
    t.business_name AS target_name,
    s1.business_address AS s1_address,
    t.business_address AS target_address

FROM pairs p

JOIN read_csv_auto(
    '{FILES["train_source1"].as_posix()}',
    delim='\t'
) s1
    ON p.source1_entity_id = s1.entity_id

JOIN targets t
    ON p.matched_entity_id = t.entity_id
"""

sample_pairs = con.execute(sample_query).fetchdf()

print("Sampled true pairs:", len(sample_pairs))
display(sample_pairs.head())

Sampled true pairs: 73600


,source1_entity_id,matched_entity_id,s1_name,target_name,s1_address,target_address
0,S1-341667260,S2-272338010,Interstate Mountain Paloma LLC,Interstate Mountain Páloma LLC,"107 Maddox Drive, Orange Town, VA","107 1/2 MADDOX DR, ORANGE TOWN, VA"
1,S1-483584433,S2-436265775,KZY Handicraft,KZY Center,"Aristo 1404, 34 Pavillion, Sector 34, Noida, G...","उत्तर प्रदेश, ARISTO 1404 , 34 PAVILLION, SECT..."
2,S1-818568031,S2-853939785,Madden Industries Inc,Madden Industries,"5435 Meadowlands Terrace, Portland, OR","5435 MEADOWLANDS TER, PORTLAND, OR"
3,S1-101765943,S2-162040631,"Pinnacle Wireless Services, LLC","Llc Pinnade Wireless Services,","920 Lakepoint Drive, Unit APT 706, Augusta, KS","920 LAKEPOINT DRIVE, AUGUSTA, KS"
4,S1-931926054,S2-236617143,Reliable Software Dynamics,RELIABLE DYNAMICS S0FTWARE,"24 Marilyn Road, Waterford, CT","MARILYN RD, WATERFORD, CT"


In [6]:
sample_pairs["s1_name_norm"] = (
    sample_pairs["s1_name"].apply(normalize_name)
)

sample_pairs["target_name_norm"] = (
    sample_pairs["target_name"].apply(normalize_name)
)

sample_pairs["s1_address_norm"] = (
    sample_pairs["s1_address"].apply(normalize_address)
)

sample_pairs["target_address_norm"] = (
    sample_pairs["target_address"].apply(normalize_address)
)

In [7]:
name_raw = (
    sample_pairs["s1_name"]
    .fillna("")
    .eq(sample_pairs["target_name"].fillna(""))
)

name_norm = (
    sample_pairs["s1_name_norm"]
    .eq(sample_pairs["target_name_norm"])
    & sample_pairs["s1_name_norm"].notna()
)

addr_raw = (
    sample_pairs["s1_address"]
    .eq(sample_pairs["target_address"])
    & sample_pairs["s1_address"].notna()
    & sample_pairs["target_address"].notna()
)

addr_norm = (
    sample_pairs["s1_address_norm"]
    .eq(sample_pairs["target_address_norm"])
    & sample_pairs["s1_address_norm"].notna()
    & sample_pairs["target_address_norm"].notna()
)

print(
    "Raw name exact:       ",
    round(name_raw.mean() * 100, 2),
    "%"
)

print(
    "Normalized name exact:",
    round(name_norm.mean() * 100, 2),
    "%"
)

print(
    "Raw address exact:       ",
    round(addr_raw.mean() * 100, 2),
    "%"
)

print(
    "Normalized address exact:",
    round(addr_norm.mean() * 100, 2),
    "%"
)

Raw name exact:        4.62 %
Normalized name exact: 15.82 %
Raw address exact:        2.16 %
Normalized address exact: 7.26 %


In [8]:
improved_name = sample_pairs[
    (~name_raw)
    & name_norm
][[
    "s1_name",
    "target_name",
    "s1_name_norm",
    "target_name_norm"
]]

print("Name pairs improved by normalization:", len(improved_name))

display(improved_name.head(20))

Name pairs improved by normalization: 8244


,s1_name,target_name,s1_name_norm,target_name_norm
9,Trading Biswas Services Pvt Ltd,Trading Biswas Services Pvt Ltd,trading biswas services pvt ltd,trading biswas services pvt ltd
10,Solstice LLC,SOLSTICE LLC,solstice llc,solstice llc
20,"Clark, Sines and Stowell LLC","Clark, Sines And Stowell LLC","clark, sines and stowell llc","clark, sines and stowell llc"
26,Golden Aldel LLC,Golden Aldel Llc,golden aldel llc,golden aldel llc
33,Vasquez & Garcia Sunrise PLLC,VASQUEZ & GARCIA SUNRISE PLLC,vasquez & garcia sunrise pllc,vasquez & garcia sunrise pllc
46,Dev Holdings,Dev Holdings,dev holdings,dev holdings
49,Coastal Council Inc,COASTAL COUNCIL INC,coastal council inc,coastal council inc
52,KD Can,KD Can,kd can,kd can
53,Advanced Marine Industries,ADVANCED MARINE INDUSTRIES,advanced marine industries,advanced marine industries
57,"Datao, Inc","DATAO, INC","datao, inc","datao, inc"


In [9]:
improved_address = sample_pairs[
    (~addr_raw)
    & addr_norm
][[
    "s1_address",
    "target_address",
    "s1_address_norm",
    "target_address_norm"
]]

print(
    "Address pairs improved by normalization:",
    len(improved_address)
)

display(improved_address.head(20))

Address pairs improved by normalization: 3749


,s1_address,target_address,s1_address_norm,target_address_norm
8,"7246 13, Kulm, ND","7246 13, KULM, ND","7246 13, kulm, nd","7246 13, kulm, nd"
14,"431 Anemone Avenue, Village Of Palmyra, WI","431 ANEMONE AVENUE, VILLAGE OF PALMYRA, WI","431 anemone avenue, village of palmyra, wi","431 anemone avenue, village of palmyra, wi"
34,"68 Hillsboro Viola Road, Hillsboro, TN","68 HILLSBORO VIOLA ROAD, HILLSBORO, TN","68 hillsboro viola road, hillsboro, tn","68 hillsboro viola road, hillsboro, tn"
37,"5745 Broadway, Chicago, IL","5745 Broadway, CHICAGO, IL","5745 broadway, chicago, il","5745 broadway, chicago, il"
44,"5162 Fm 2149, Maud, TX","5162 FM 2149, MAUD, TX","5162 fm 2149, maud, tx","5162 fm 2149, maud, tx"
68,"69, Padmavati Colony, Indore, Madhya Pradesh","69, PADMAVATI COLONY, INDORE, Madhya Pradesh","69, padmavati colony, indore, madhya pradesh","69, padmavati colony, indore, madhya pradesh"
92,"No 13/6, Thiyagarajan Street, Tsr Nagar Thiruv...","NO 13/6, THIYAGARAJAN STREET, TSR NAGAR THIRUV...","no 13/6, thiyagarajan street, tsr nagar thiruv...","no 13/6, thiyagarajan street, tsr nagar thiruv..."
133,"Fa-357 Mansarover Garden, New Delhi, North Del...","FA-357 MANSAROVER GARDEN, NEW DELHI, NORTH DEL...","fa-357 mansarover garden, new delhi, north del...","fa-357 mansarover garden, new delhi, north del..."
142,"S No. 151/1/1, Morwadi, Nr. Waghere Empire, Pi...","S NO. 151/1/1, MORWADI, NR. WAGHERE EMPIRE, PI...","s no. 151/1/1, morwadi, nr. waghere empire, pi...","s no. 151/1/1, morwadi, nr. waghere empire, pi..."
147,"11491 Spring Street, Adelanto, CA","11491 SPRING STREET, ADELANTO, CA","11491 spring street, adelanto, ca","11491 spring street, adelanto, ca"


In [10]:
import pandas as pd

test_df = con.execute(f"""
    SELECT business_name, business_address, country
    FROM read_csv_auto(
        '{FILES["train_source3"].as_posix()}',
        delim='\t'
    )
    WHERE business_address IS NULL
    LIMIT 100
""").fetchdf()

print("Rows sampled with NULL address:", len(test_df))

if len(test_df) > 0:
    print("Python types:")
    print(test_df.dtypes)

    print("\nUnique Python value types in business_address:")
    print(
        test_df["business_address"]
        .map(type)
        .value_counts()
    )

    print("\nPandas NA count:")
    print(test_df["business_address"].isna().sum())

Rows sampled with NULL address: 100
Python types:
business_name          str
business_address    object
country                str
dtype: object

Unique Python value types in business_address:
business_address
<class 'NoneType'>    100
Name: count, dtype: int64

Pandas NA count:
100


In [11]:
import re

def normalize_address_compact(value):
    value = normalize_address(value)

    if value is None:
        return None

    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

In [12]:
sample_pairs["s1_address_compact"] = (
    sample_pairs["s1_address"]
    .apply(normalize_address_compact)
)

sample_pairs["target_address_compact"] = (
    sample_pairs["target_address"]
    .apply(normalize_address_compact)
)

In [13]:
addr_compact = (
    sample_pairs["s1_address_compact"]
    .eq(sample_pairs["target_address_compact"])
    & sample_pairs["s1_address_compact"].notna()
    & sample_pairs["target_address_compact"].notna()
)

print(
    "Normalized address exact:",
    round(addr_norm.mean() * 100, 2),
    "%"
)

print(
    "Compact address exact:",
    round(addr_compact.mean() * 100, 2),
    "%"
)

Normalized address exact: 7.26 %
Compact address exact: 8.05 %


In [14]:
improved_compact = sample_pairs[
    (~addr_norm)
    & addr_compact
][[
    "s1_address",
    "target_address",
    "s1_address_norm",
    "target_address_norm",
    "s1_address_compact",
    "target_address_compact"
]]

print(
    "Pairs improved by removing punctuation:",
    len(improved_compact)
)

display(improved_compact.head(20))

Pairs improved by removing punctuation: 586


,s1_address,target_address,s1_address_norm,target_address_norm,s1_address_compact,target_address_compact
18,"12 Thistle Lane, Weymouth, MA","#12 THISTLE LANE, WEYMOUTH, MA","12 thistle lane, weymouth, ma","#12 thistle lane, weymouth, ma",12 thistle lane weymouth ma,12 thistle lane weymouth ma
32,"G-9 3Rd Floor Maharani Bagh New Delhi, New Del...","G-9- 3RD FLOOR MAHARANI BAGH NEW DELHI, NEW DE...","g-9 3rd floor maharani bagh new delhi, new del...","g-9- 3rd floor maharani bagh new delhi, new de...",g 9 3rd floor maharani bagh new delhi new delh...,g 9 3rd floor maharani bagh new delhi new delh...
95,"5704 Addison Street, Chicago, IL","5704. Addison Street, CHICAGO, IL","5704 addison street, chicago, il","5704. addison street, chicago, il",5704 addison street chicago il,5704 addison street chicago il
231,"73 500, Kokomo, IN","#73 500, KOKOMO, IN","73 500, kokomo, in","#73 500, kokomo, in",73 500 kokomo in,73 500 kokomo in
334,"23418 Partridge Road, Pretty Prairie, KS","##23418 PARTRIDGE ROAD, PRETTY PRAIRIE, KS","23418 partridge road, pretty prairie, ks","##23418 partridge road, pretty prairie, ks",23418 partridge road pretty prairie ks,23418 partridge road pretty prairie ks
373,"103 Cherry Laurel Street, Huffman, TX","103- CHERRY LAUREL STREET, HUFFMAN, TX","103 cherry laurel street, huffman, tx","103- cherry laurel street, huffman, tx",103 cherry laurel street huffman tx,103 cherry laurel street huffman tx
588,"B-1244 Indira Nagar, Lucknow, Uttar Pradesh","B-#1244 INDIRA NAGAR, LUCKNOW, Uttar Pradesh","b-1244 indira nagar, lucknow, uttar pradesh","b-#1244 indira nagar, lucknow, uttar pradesh",b 1244 indira nagar lucknow uttar pradesh,b 1244 indira nagar lucknow uttar pradesh
633,"4 Windstone Drive, North Little Rock, AR","4- WINDSTONE DRIVE, NORTH LITTLE ROCK, AR","4 windstone drive, north little rock, ar","4- windstone drive, north little rock, ar",4 windstone drive north little rock ar,4 windstone drive north little rock ar
787,"A-5, Ground Floor, Gaur Cascades, Ghaziabad, U...","A-#5, GROUND FLOOR, GAUR CASCADES, GHAZIABAD, ...","a-5, ground floor, gaur cascades, ghaziabad, u...","a-#5, ground floor, gaur cascades, ghaziabad, ...",a 5 ground floor gaur cascades ghaziabad uttar...,a 5 ground floor gaur cascades ghaziabad uttar...
876,"372 Nolan Lane, Bel Air, MD","#372 NOLAN LANE, BEL AIR, MD","372 nolan lane, bel air, md","#372 nolan lane, bel air, md",372 nolan lane bel air md,372 nolan lane bel air md


In [15]:
missing_tokens = [
    "",
    "na",
    "n/a",
    "n.a.",
    "nan",
    "null",
    "none",
    "unknown",
    "-",
    "--",
]

columns = [
    "business_name",
    "business_address",
    "country",
]

for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3",
]:
    path = FILES[name]

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    for column in columns:
        query = f"""
        SELECT
            CASE
                WHEN {column} IS NULL THEN '<SQL_NULL>'
                WHEN TRIM({column}) = '' THEN '<EMPTY>'
                ELSE LOWER(TRIM({column}))
            END AS value,
            COUNT(*) AS count
        FROM read_csv_auto(
            '{path.as_posix()}',
            delim='\\t'
        )
        GROUP BY 1
        """

        df = con.execute(query).fetchdf()

        suspicious = df[
            df["value"].isin(
                ["<SQL_NULL>", "<EMPTY>"] + missing_tokens
            )
        ].sort_values("count", ascending=False)

        print(f"\n{column}")
        display(suspicious)


train_source1

business_name


,value,count



business_address


,value,count



country


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


business_address


,value,count
1815264,<SQL_NULL>,136098



country


,value,count


In [16]:
path = FILES["train_source3"]

query = f"""
SELECT
    business_address,
    COUNT(*) AS count
FROM read_csv_auto(
    '{path.as_posix()}',
    delim='\\t'
)
WHERE business_address IS NULL
   OR LOWER(TRIM(business_address)) IN (
        '',
        'na',
        'n/a',
        'n.a.',
        'nan',
        'null',
        'none',
        'unknown',
        '-',
        '--'
   )
GROUP BY business_address
ORDER BY count DESC
"""

display(con.execute(query).fetchdf())

,business_address,count
0,None,175916


In [17]:
query = f"""
SELECT
    LOWER(TRIM(business_address)) AS value,
    COUNT(*) AS count
FROM read_csv_auto(
    '{FILES["train_source3"].as_posix()}',
    delim='\\t'
)
WHERE business_address IS NOT NULL
  AND (
       LOWER(business_address) LIKE '%unknown%'
    OR LOWER(business_address) LIKE '%not available%'
    OR LOWER(business_address) LIKE '%not provided%'
    OR LOWER(business_address) LIKE '%no address%'
    OR LOWER(business_address) = 'nil'
  )
GROUP BY 1
ORDER BY count DESC
LIMIT 50
"""

display(con.execute(query).fetchdf())

,value,count
0,"112 unknown, elizabeth, arkansas",2
1,"#216 unknown, unit a, cardington, ohio",2
2,"chickasha, oklahoma, 2119 unknown rd",1
3,"#216 unknown, # a, cardington, ohio",1
4,"943 unknown avenue, chickasha, oklahoma",1
5,"#216 unknown, cardington, ohio",1
6,"arizona, kirkland, 16615 unknown mexican avenue",1
7,"2126-2128 unknown rd, chickasha, oklahoma",1
8,"124 unknown road, killington, vermont",1
9,"22035 unknown, branchville, indiana",1


In [18]:
missing_name_tokens = [
    "na",
    "nan",
    "null",
]

for name in [
    "train_source2",
    "train_source3",
    "test_source2",
    "test_source3",
]:
    path = FILES[name]

    query = f"""
    SELECT
        entity_id,
        business_name,
        country,
        business_address
    FROM read_csv_auto(
        '{path.as_posix()}',
        delim='\t'
    )
    WHERE LOWER(TRIM(business_name)) IN (
        'na',
        'nan',
        'null'
    )
    ORDER BY LOWER(TRIM(business_name)), entity_id
    """

    result = con.execute(query).fetchdf()

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    display(result)


train_source2


,entity_id,business_name,country,business_address
0,S2-102598836,Na,US,"9-B CLINTON ST, EASTHAMPTON, MA"
1,S2-391540425,Na,US,"3 VILLAGE VIEW, FREEPORT, ME"
2,S2-599473918,NA,US,"BROOKHAVEN, 7 LILAC LANE, NY"
3,S2-776053769,Na,US,"WEBB CITY, 114 MSIN ST, MO"
4,S2-982925237,NA,US,"11 OX YOKE CIRCLE, EAST HAMPTON, CT"
5,S2-876292278,NAN,US,"1305 3610, SPANISH FORK, UT"



train_source3


,entity_id,business_name,country,business_address
0,S3-203011842,Na,US,"Blue Ridge Pl, # Unit 215, Altoona, Iowa"
1,S3-207014979,NA,India,"C-2 Mansarovar Riico Industrial Area, Jaipur, ..."
2,S3-263823274,NA,US,"519 Odeneal Street, Dallas, TX"
3,S3-382185873,NA,India,"Kh No. 138, 1St Floor, Vill: Nawada, Vipin Gar..."
4,S3-391601813,NA,US,"209 3rd Street, KY, Hardinsburg"
5,S3-42389827,NA,India,"137 - R, Industrial Area B, Near Pahwa Hospita..."
6,S3-508022313,Na,US,"1449 Sapphire Lane, Waukee, Iowa"
7,S3-615108446,Na,US,"412 Berlin Pvt Rd, Gallatin, Tennessee"
8,S3-617170111,NA,US,"120 Shore Drive, Jarvisburg, North Carolina"
9,S3-640593334,NA,India,"261/12 F/F Amritpuri B Garhi Jharia Maria, Sou..."



test_source2


,entity_id,business_name,country,business_address
0,S2-109029910,NA,France,"45 RUE DES DOUVES, Bordeaux, Gironde"
1,S2-10975227,NA,France,"15 RUE ARMAED MEESCHAERT, Roubaix"
2,S2-156787089,NA,France,"NO 20 RUE GABRIEL DUPUY, Mérignac, Nouvelle-Aq..."
3,S2-16141727,NA,France,"RUE DU FOUR A CHAUX, Lille"
4,S2-165494875,NA,France,"2 RUE SOUBZMAIN, Nantes, Pays de la Loire"
5,S2-17593083,NA,France,"Nº 119 BOULEVARD ERNEST DALBY, Nantes, Loire-A..."
6,S2-189780924,NA,France,"LE BIS D' ACHELLES, Tourcoing, Hauts-de-France"
7,S2-194797905,NA,France,"18 R. DU CHEVAL GRIS, Calais, Hauts-de-France"
8,S2-210220541,NA,France,"3 CH DE LA SÉCHERIE, Nantes, Loire-Atlantique"
9,S2-237980251,Na,US,"SAINT LOUIS PARK, 165 KILMER AVE, MN"



test_source3


,entity_id,business_name,country,business_address
0,S3-131417344,NA,France,"1 Cour Saint-dominique, 43 Rue De Cartigny, Ro..."
1,S3-137727332,NA,India,"F-204 Samsara Luxury Appt, Chhani Jakat Naka T..."
2,S3-166772876,NA,France,"3 R. De Lesticaire, Pessac, Nouvelle-Aquitaine"
3,S3-167591643,NA,France,"80 Rue Félix Ménétrier, Nantes, Pays de la Loire"
4,S3-184111661,NA,France,"Hauts-de-France, Dunkerque, 16 Rue de la Plage"
...,...,...,...,...
56,S3-959971245,NA,France,"33 R Van Dyck, Roubaix"
57,S3-964889976,NA,France,"3 Rmelle Lilly Reich, Nantes"
58,S3-994570961,NA,India,"E 3/11 Ganga Puramviman Nagar, Pune, Maharashtra"
59,S3-996535728,NA,France,"Pays de la Loire, 546 Route De Saint Josph, Na..."


In [19]:
for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3",
]:
    path = FILES[name]

    query = f"""
    SELECT
        LOWER(TRIM(business_name)) AS value,
        COUNT(*) AS count
    FROM read_csv_auto(
        '{path.as_posix()}',
        delim='\\t'
    )
    WHERE business_name IS NOT NULL
      AND TRIM(business_name) <> ''
      AND LENGTH(TRIM(business_name)) <= 12
    GROUP BY 1
    ORDER BY count DESC
    LIMIT 100
    """

    result = con.execute(query).fetchdf()

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    display(result)


train_source1


,value,count
0,eye group,207
1,dental group,205
2,family group,204
3,vision group,199
4,meridian,182
...,...,...
95,cinder llc,85
96,northwind,85
97,vertex llc,84
98,birch llc,84



train_source2


,value,count
0,primary care,397
1,urgent care,366
2,the dent,246
3,main street,237
4,foot & ankle,235
...,...,...
95,cedar llc,72
96,emerald,72
97,atlas,72
98,vega llc,72



train_source3


...

,value,count
0,eye group,110
1,shree & co,97
2,family group,93
3,dental group,92
4,meridian,92
...,...,...
95,osprey,43
96,sequoia llc,43
97,sirius llc,43
98,willow llc,43



test_source2


,value,count
0,cc,302
1,pc,197
2,urgent care,197
3,lc,194
4,primary care,177
...,...,...
95,family group,50
96,vertex,50
97,ns,50
98,calais ecole,50



test_source3


,value,count
0,cc,387
1,sc,358
2,ac,297
3,pc,286
4,lc,245
...,...,...
95,bf,86
96,sm,85
97,big,85
98,df,85


In [20]:
suspect_words = [
    "na",
    "nan",
    "null",
    "none",
    "nil",
    "missing",
    "unknown",
    "unavailable",
    "not available",
    "not provided",
    "n/a",
]

for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3",
]:
    path = FILES[name]

    conditions = " OR ".join(
        f"LOWER(TRIM(business_name)) = '{word}'"
        for word in suspect_words
    )

    query = f"""
    SELECT
        LOWER(TRIM(business_name)) AS value,
        COUNT(*) AS count
    FROM read_csv_auto(
        '{path.as_posix()}',
        delim='\\t'
    )
    WHERE business_name IS NOT NULL
      AND ({conditions})
    GROUP BY 1
    ORDER BY count DESC
    """

    result = con.execute(query).fetchdf()

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    display(result)


train_source1


,value,count


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


train_source2


,value,count
0,na,5
1,nan,1


...


test_source2


,value,count
0,na,49


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


test_source3


,value,count
0,na,60
1,null,1


In [21]:
for name in [
    "train_source1",
    "train_source2",
    "train_source3",
]:
    path = FILES[name]

    query = f"""
    WITH data AS (
        SELECT
            entity_id,
            business_name
        FROM read_csv_auto(
            '{path.as_posix()}',
            delim='\t'
        )
    )
    SELECT
        LOWER(TRIM(business_name)) AS name_norm,
        COUNT(*) AS records
    FROM data
    WHERE business_name IS NOT NULL
      AND TRIM(business_name) <> ''
    GROUP BY 1
    HAVING COUNT(*) > 1
    ORDER BY records DESC
    LIMIT 20
    """

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    display(con.execute(query).fetchdf())


train_source1


,name_norm,records
0,primary care group,253
1,ear nose & throat group,251
2,pediatric group,222
3,womens health group,220
4,physical therapy group,218
5,pediatric dental group,216
6,behavioral health group,215
7,chiropractic group,209
8,orthopedic group,208
9,eye group,207



train_source2


,name_norm,records
0,primary care,397
1,physical therapy,386
2,womens health,369
3,urgent care,366
4,behavioral health,363
5,earnosethroat.com,348
6,internal medicine,346
7,pediatric dental,337
8,pediatric dentistry,335
9,ear nose & throat,253



train_source3


,name_norm,records
0,primary care,421
1,physical therapy,399
2,pediatric dental,393
3,womens health,379
4,urgent care,377
5,pediatric dentistry,361
6,behavioral health,330
7,pediatric,321
8,internal medicine,319
9,earnosethroat.com,307


In [24]:
tests = [
    "CC",
    "PC",
    "LC",
    "IBM",
    "Na Services",
    "Banana",
]

for x in tests:
    print(repr(x), "->", repr(normalize_name(x)))

'CC' -> 'cc'
'PC' -> 'pc'
'LC' -> 'lc'
'IBM' -> 'ibm'
'Na Services' -> 'na services'
'Banana' -> 'banana'


In [22]:
tests = [
    None,
    float("nan"),
    pd.NA,
    "",
    "   ",
    "NA",
    "NAN",
    "NULL",
]

for x in tests:
    print(repr(x), "->", repr(normalize_name(x)))

None -> None
nan -> None
<NA> -> None
'' -> None
'   ' -> None
'NA' -> None
'NAN' -> None
'NULL' -> None


In [23]:
address_tests = [
    "256 56th Avenue, Anchorage, AK",
    "256. 56TH AVENUE, ANCHORAGE, AK",
    "14 Black Mountain Road, N/A, Encinal, New Mexico",
    "112 unknown, Elizabeth, Arkansas",
    None,
]

for x in address_tests:
    print("RAW    :", repr(x))
    print("NORMAL :", repr(normalize_address(x)))
    print("COMPACT:", repr(normalize_address_compact(x)))
    print()

RAW    : '256 56th Avenue, Anchorage, AK'
NORMAL : '256 56th avenue, anchorage, ak'
COMPACT: '256 56th avenue anchorage ak'

RAW    : '256. 56TH AVENUE, ANCHORAGE, AK'
NORMAL : '256. 56th avenue, anchorage, ak'
COMPACT: '256 56th avenue anchorage ak'

RAW    : '14 Black Mountain Road, N/A, Encinal, New Mexico'
NORMAL : '14 black mountain road, n/a, encinal, new mexico'
COMPACT: '14 black mountain road n a encinal new mexico'

RAW    : '112 unknown, Elizabeth, Arkansas'
NORMAL : '112 unknown, elizabeth, arkansas'
COMPACT: '112 unknown elizabeth arkansas'

RAW    : None
NORMAL : None
COMPACT: None

